# 直近得点者モデル

Dixon & Robinson (1998) のモデルVIをもとに、スコアによる倍率の区分を「リード・同点・ビハインド」と「直前に点を取ったのが自分か相手か」に変えたモデル。ほかの部分（α, β, γ_h, ρ, ξ）はモデルVIと同じ。

### 状態の区分（そのチームから見た7区分）
| 記号 | 状態 |
|---|---|
| `00` | 0-0（基準、倍率 = 1） |
| `lead_scored` | リード中で、直前の得点が自分 |
| `lead_conceded` | リード中で、直前の得点が相手 |
| `behind_scored` | ビハインド中で、直前の得点が自分 |
| `behind_conceded` | ビハインド中で、直前の得点が相手 |
| `level_scored` | 同点（0-0以外）で、直前の得点が自分 |
| `level_conceded` | 同点（0-0以外）で、直前の得点が相手 |

「直前」は時間では区切らず、最後に点を取ったのがどちらかで決める。

## データ

`dixon_robinson_model.ipynb` と同じデータを使う。退場の行は使わず、0-0の試合は「得点なしの試合」として扱う。

In [13]:
import glob
import os

import numpy as np
import pandas as pd
from scipy.optimize import minimize

## モデル

$$\lambda(t) = \rho(t)\big(\alpha_i\beta_j\gamma_h \cdot \theta_{s_H(t)} + \xi_1 t\big)$$
$$\mu(t) = \rho(t)\big(\alpha_j\beta_i \cdot \theta_{s_A(t)} + \xi_2 t\big)$$

- $s_H(t)$, $s_A(t)$ は、時刻 $t$ にホーム・アウェイから見た状態
- 0-0 の倍率を1とする（$\theta_{00}=1$）
- 倍率 $\theta$ はホームとアウェイで同じものを使う

ホームの状態が決まればアウェイの状態も決まる（例：ホームが「リード中・直前の得点が自分」なら、アウェイは「ビハインド中・直前の得点が相手」）。

In [14]:
# インジュリータイム（アディショナルタイム）とみなす区間の境界（元のモデルと同じ）
INJ1_START, INJ1_END = 44 / 90, 45 / 90
INJ2_START, INJ2_END = 89 / 90, 90 / 90

# 基準状態 "00" 以外の6状態（この順でパラメータを並べる）
STATES = [
    "lead_scored", "lead_conceded",
    "behind_scored", "behind_conceded",
    "level_scored", "level_conceded",
]
N_STATES = len(STATES)


class _InvalidRate(Exception):
    # 得点強度が非正になってしまった場合に投げる内部例外（最適化の枝刈り用）
    pass


def _team_states(x, y, last):
    '''
    スコア (x, y) と最後に得点した側 last ("H" / "A" / None) から、
    (ホームから見た状態, アウェイから見た状態) を返す。
    '''
    if last is None:              # まだ誰も得点していない = 0-0
        return "00", "00"
    diff = x - y
    if diff > 0:
        h_rel, a_rel = "lead", "behind"
    elif diff < 0:
        h_rel, a_rel = "behind", "lead"
    else:
        h_rel, a_rel = "level", "level"
    if last == "H":
        return f"{h_rel}_scored", f"{a_rel}_conceded"
    return f"{h_rel}_conceded", f"{a_rel}_scored"

## データの前処理

csvを、試合ごとの得点の並びに変える。

In [15]:
def prepare_matches(df):
    '''
    Goals_and_red_cards の df を、試合ごとのイベント列に変換する（退場者イベントは無視）。
    戻り値: list of {"home", "away", "events": [(t, "H_GOAL"/"A_GOAL"), ...]}
    '''
    matches = {}
    d = df.sort_values(["match_id", "dr_time"])
    for row in d.itertuples(index=False):
        mid = row.match_id
        if mid not in matches:
            matches[mid] = {"home": row.home_team, "away": row.away_team, "events": []}
        if pd.isna(row.dr_time):
            continue      # 0-0 の試合のプレースホルダー行
        if pd.notna(row.red_card):
            continue      # 退場者イベントは使わない
        t = float(row.dr_time)
        kind = "H_GOAL" if int(row.home_away) == 0 else "A_GOAL"
        matches[mid]["events"].append((t, kind))
    return list(matches.values())


def build_index(matches):
    teams = sorted(set([m["home"] for m in matches] + [m["away"] for m in matches]))
    idx = {t: i for i, t in enumerate(teams)}
    return teams, idx

## 状態ごとのデータ数

「リード中・直前の得点が相手」などは2点以上入らないと出てこないので、データが少ない状態がある。
推定の前に、状態ごとに、その状態だった時間（1試合 = 1）と得点数を数えておく。

In [16]:
def state_occupancy(matches):
    '''
    各状態について、その状態にいたチームの延べ滞在時間（1 = 1試合分）と、
    その状態のチームが挙げた得点数を集計する（ホーム・アウェイ別）。
    '''
    rows = {}
    def add(state, side, dt, goal):
        r = rows.setdefault((state, side), {"state": state, "side": side, "exposure": 0.0, "goals": 0})
        r["exposure"] += dt
        r["goals"] += goal

    for m in matches:
        x = y = 0
        last = None
        t_prev = 0.0
        for t_ev, kind in sorted(m["events"], key=lambda e: e[0]):
            sh, sa = _team_states(x, y, last)
            add(sh, "home", t_ev - t_prev, int(kind == "H_GOAL"))
            add(sa, "away", t_ev - t_prev, int(kind == "A_GOAL"))
            if kind == "H_GOAL":
                x += 1; last = "H"
            else:
                y += 1; last = "A"
            t_prev = t_ev
        sh, sa = _team_states(x, y, last)
        add(sh, "home", 1.0 - t_prev, 0)
        add(sa, "away", 1.0 - t_prev, 0)

    out = pd.DataFrame(rows.values())
    out["goals_per_match"] = out["goals"] / out["exposure"].where(out["exposure"] > 0)
    order = {s: i for i, s in enumerate(["00"] + STATES)}
    out = out.sort_values(["side", "state"], key=lambda c: c.map(order) if c.name == "state" else c)
    return out.reset_index(drop=True)

## パラメータと尤度関数

チーム以外のパラメータは `gamma_h, rho1, rho2, theta（6状態）, xi1, xi2` の11個。
正の値でないといけないもの（α, β, γ_h, ρ, θ）は `exp()` をかけて正にする。

In [17]:
PARAM_NAMES_GLOBAL = ["gamma_h", "rho1", "rho2"] + [f"theta_{s}" for s in STATES] + ["xi1", "xi2"]
N_GLOBAL = len(PARAM_NAMES_GLOBAL)


def unpack_params(theta, n_teams):
    a = theta[0:n_teams]
    b = theta[n_teams:2 * n_teams]
    g = theta[2 * n_teams:]

    alpha = np.exp(a)
    alpha = alpha / alpha.mean()      # 制約 mean(alpha) = 1
    beta = np.exp(b)

    gamma_h, rho1, rho2 = np.exp(g[0]), np.exp(g[1]), np.exp(g[2])
    th = {"00": 1.0}
    for k, s in enumerate(STATES):
        th[s] = np.exp(g[3 + k])
    xi1, xi2 = g[-2], g[-1]
    return alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2

In [18]:
def _check_pos(base, xi, t):
    if base + xi * t <= 0:
        raise _InvalidRate()


def _rho_at(t, rho1, rho2):
    if INJ1_START < t <= INJ1_END:
        return rho1
    if INJ2_START < t <= INJ2_END:
        return rho2
    return 1.0


def _integrate_rate(t1, t2, base, xi, rho1, rho2):
    # 区間 [t1, t2]（状態一定）で ∫ rho(t)*(base + xi*t) dt
    bpoints = sorted({t1, t2} | {p for p in (INJ1_START, INJ1_END, INJ2_START, INJ2_END) if t1 < p < t2})
    for p in bpoints:
        _check_pos(base, xi, p)
    total = 0.0
    for a, c in zip(bpoints[:-1], bpoints[1:]):
        r = _rho_at(0.5 * (a + c), rho1, rho2)
        total += r * (base * (c - a) + xi * (c ** 2 - a ** 2) / 2.0)
    return total


def _rate_at(t, base, xi, rho1, rho2):
    val = base + xi * t
    if val <= 0:
        raise _InvalidRate()
    return _rho_at(t, rho1, rho2) * val


def _match_loglik(match, idx, alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2):
    hi, aj = idx[match["home"]], idx[match["away"]]
    lam_k = alpha[hi] * beta[aj] * gamma_h   # home側の基礎強度（0-0時）
    mu_k = alpha[aj] * beta[hi]              # away側の基礎強度（0-0時）

    x = y = 0
    last = None
    t_prev = 0.0
    ll = 0.0
    for t_ev, kind in sorted(match["events"], key=lambda e: e[0]):
        sh, sa = _team_states(x, y, last)
        base_h = lam_k * th[sh]
        base_a = mu_k * th[sa]
        ll -= _integrate_rate(t_prev, t_ev, base_h, xi1, rho1, rho2)
        ll -= _integrate_rate(t_prev, t_ev, base_a, xi2, rho1, rho2)
        if kind == "H_GOAL":
            ll += np.log(_rate_at(t_ev, base_h, xi1, rho1, rho2))
            x += 1; last = "H"
        else:
            ll += np.log(_rate_at(t_ev, base_a, xi2, rho1, rho2))
            y += 1; last = "A"
        t_prev = t_ev

    sh, sa = _team_states(x, y, last)
    ll -= _integrate_rate(t_prev, 1.0, lam_k * th[sh], xi1, rho1, rho2)
    ll -= _integrate_rate(t_prev, 1.0, mu_k * th[sa], xi2, rho1, rho2)
    return ll


def negative_log_likelihood(theta, matches, idx):
    params = unpack_params(theta, len(idx))
    total = 0.0
    for m in matches:
        try:
            total += _match_loglik(m, idx, *params)
        except _InvalidRate:
            total += -1e9   # 強度が負になる領域には大きなペナルティ
    if not np.isfinite(total):
        return 1e12
    return -total


def _count_invalid_matches(theta, matches, idx):
    params = unpack_params(theta, len(idx))
    n_invalid = 0
    for m in matches:
        try:
            _match_loglik(m, idx, *params)
        except _InvalidRate:
            n_invalid += 1
    return n_invalid

## 初期値

チームごとの平均得点・平均失点から、攻撃力と守備力の初期値を作る。倍率 θ は1から始める。

In [19]:
def initial_theta(matches, idx):
    n_teams = len(idx)
    goals_for = np.zeros(n_teams)
    goals_against = np.zeros(n_teams)
    games = np.zeros(n_teams)
    total_goals = total_games = home_goals_total = away_goals_total = 0
    for m in matches:
        hi, aj = idx[m["home"]], idx[m["away"]]
        xg = sum(1 for t, k in m["events"] if k == "H_GOAL")
        yg = sum(1 for t, k in m["events"] if k == "A_GOAL")
        goals_for[hi] += xg; goals_against[aj] += xg
        goals_for[aj] += yg; goals_against[hi] += yg
        games[hi] += 1; games[aj] += 1
        total_goals += xg + yg; total_games += 1
        home_goals_total += xg; away_goals_total += yg

    avg = total_goals / max(2 * total_games, 1)
    att0 = np.clip(np.where(games > 0, (goals_for / np.maximum(games, 1)) / avg, 1.0), 0.3, 3.0)
    def0 = np.clip(np.where(games > 0, (goals_against / np.maximum(games, 1)) / avg, 1.0), 0.3, 3.0)

    g0 = np.zeros(N_GLOBAL)
    g0[0] = np.log(max(home_goals_total / max(away_goals_total, 1), 0.1))
    return np.concatenate([np.log(att0), np.log(def0), g0])


def make_bounds(n_teams):
    bounds = [(-3, 3)] * (2 * n_teams)
    bounds += [(-3, 3)] * 3          # gamma_h, rho1, rho2
    bounds += [(-3, 3)] * N_STATES   # 状態倍率（logスケール）
    bounds += [(-5, 5), (-5, 5)]     # xi1, xi2
    return bounds

## 推定

L-BFGS-B → Powell の2段階で最適化する。
途中で得点強度が負になるところに入って値が悪くなることがあるので、最後は「初期値・1段目・2段目」のうち一番尤度が高いものを使う。

In [20]:
def fit_recent_scorer(df, theta_init=None, maxiter=1000, disp=False, verbose=True):
    '''
    直近得点者モデルを推定する。

    Parameters
    ----------
    df : pandas.DataFrame   1リーグ・1シーズン分の Goals_and_red_cards
    theta_init : ndarray    初期値（None なら initial_theta）

    Returns
    -------
    summary, team_params, global_params, state_table, res
    '''
    matches = prepare_matches(df)
    teams, idx = build_index(matches)
    n_teams = len(teams)

    theta0 = initial_theta(matches, idx) if theta_init is None else np.asarray(theta_init, float)
    bounds = make_bounds(n_teams)

    res1 = minimize(
        negative_log_likelihood, theta0, args=(matches, idx),
        method="L-BFGS-B", bounds=bounds,
        options={"maxiter": maxiter, "maxfun": maxiter * 50},
    )
    f0 = negative_log_likelihood(theta0, matches, idx)
    start2 = res1.x if res1.fun <= f0 else theta0
    res = minimize(
        negative_log_likelihood, start2, args=(matches, idx),
        method="Powell", bounds=bounds,
        options={"maxiter": maxiter * 20, "maxfev": maxiter * 200, "xtol": 1e-10, "ftol": 1e-12},
    )
    if disp:
        print(f"[stage1: L-BFGS-B] loglik={-res1.fun:.4f} success={res1.success}")
        print(f"[stage2: Powell]   loglik={-res.fun:.4f} success={res.success}")

    # Powell が途中で強度が負になる領域に入り、1段目より悪い値で終わることがあるため、
    # 初期値・1段目・2段目のうち最も尤度の高い点を推定値とする
    candidates = [(f0, theta0, "initial"), (res1.fun, res1.x, "L-BFGS-B"), (res.fun, res.x, "Powell")]
    best_fun, best_x, best_stage = min(candidates, key=lambda c: c[0])
    if best_stage != "Powell":
        res.x, res.fun = best_x, best_fun
        if disp:
            print(f"[selected: {best_stage}] loglik={-best_fun:.4f}")

    theta = res.x
    alpha, beta, gamma_h, rho1, rho2, th, xi1, xi2 = unpack_params(theta, n_teams)

    n_params = len(theta)
    loglik = -res.fun
    aic = 2 * n_params - 2 * loglik
    bic = n_params * np.log(len(matches)) - 2 * loglik   # n = 試合数

    team_params = pd.DataFrame({"team": teams, "alpha_attack": alpha, "beta_defence": beta})

    g = theta[2 * n_teams:]
    est = [np.exp(v) for v in g[:-2]] + [g[-2], g[-1]]
    global_params = pd.DataFrame({"parameter": PARAM_NAMES_GLOBAL, "estimate": est})

    state_table = pd.DataFrame({"state": ["00"] + STATES, "theta": [th[s] for s in ["00"] + STATES]})

    n_invalid = _count_invalid_matches(theta, matches, idx)
    message = str(res.message)
    if n_invalid > 0:
        message = (f"[WARNING] {n_invalid} match(es) still have non-positive scoring intensity; "
                   f"log-likelihood/AIC/BIC are not reliable. " + message)

    summary = {
        "n_matches": len(matches), "n_teams": n_teams, "n_params": n_params,
        "log_likelihood": loglik, "AIC": aic, "BIC": bic,
        "converged": bool(res.success), "message": message,
    }

    if verbose:
        print("==== 直近得点者モデル ====")
        print(f"試合数: {summary['n_matches']}, チーム数: {n_teams}, パラメータ数: {n_params}")
        print(f"対数尤度: {loglik:.3f}  AIC: {aic:.3f}  BIC: {bic:.3f}")
        print(f"収束: {summary['converged']} ({message})")
        print()
        print("---- チーム別パラメータ ----")
        print(team_params.to_string(index=False))
        print()
        print("---- 状態倍率 ----")
        print(state_table.to_string(index=False))
        print()
        print("---- 共通パラメータ ----")
        print(global_params.to_string(index=False))

    return summary, team_params, global_params, state_table, res

## 1ファイルで試す

Premier League 2015/16 で、状態ごとのデータ数を見てから推定する。

In [21]:
df = pd.read_csv("../statsbomb_data/Premier_League/PL2015-2016_goals_and_red_cards.csv")
state_occupancy(prepare_matches(df))

,state,side,exposure,goals,goals_per_match
0,00,away,154.463704,152,0.984050
1,lead_scored,away,72.059259,104,1.443257
2,lead_conceded,away,6.712778,9,1.340727
3,behind_scored,away,8.371667,14,1.672307
4,behind_conceded,away,101.641852,128,1.259324
5,level_scored,away,20.034074,27,1.347704
6,level_conceded,away,16.716667,25,1.495513
7,00,home,154.463704,196,1.268907
8,lead_scored,home,101.641852,178,1.751247
9,lead_conceded,home,8.371667,14,1.672307


In [22]:
summary, team_params, global_params, state_table, res = fit_recent_scorer(df, disp=True)

[stage1: L-BFGS-B] loglik=-555.5795 success=True
[stage2: Powell]   loglik=-555.5795 success=True
==== 直近得点者モデル ====
試合数: 380, チーム数: 20, パラメータ数: 51
対数尤度: -555.579  AIC: 1213.159  BIC: 1414.108
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                team  alpha_attack  beta_defence
     AFC Bournemouth      0.856906      1.087627
             Arsenal      1.342864      0.522351
         Aston Villa      0.327512      1.241396
             Chelsea      1.229386      0.771365
      Crystal Palace      0.653050      0.760162
             Everton      1.234512      0.852874
      Leicester City      1.414778      0.444690
           Liverpool      1.311778      0.715674
     Manchester City      1.523064      0.652338
   Manchester United      0.935293      0.469964
    Newcastle United      0.787288      1.057043
        Norwich City      0.698866      1.059877
         Southampton      1.171597      0.551659
          Stoke City      0.787501      0.804630
  

## 全リーグ・全シーズンで推定する

全ファイルで推定し、AIC・BICの一覧とパラメータをcsvに保存する。

In [23]:
DATA_DIR = "../statsbomb_data"
OUT_DIR = "./recent_scorer_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_goals_and_red_cards.csv"), recursive=True))
print(f"{len(csv_paths)} 件のcsvが見つかりました")

results_all = []
for path in csv_paths:
    print("=" * 60)
    print(path)
    df_i = pd.read_csv(path)
    league = df_i["competition_name"].iloc[0] if "competition_name" in df_i.columns and len(df_i) else os.path.basename(path)
    season = df_i["season_name"].iloc[0] if "season_name" in df_i.columns and len(df_i) else ""
    tag = f"{league}_{season}".replace("/", "-").replace(" ", "_")
    try:
        summary_i, tp_i, gp_i, st_i, _ = fit_recent_scorer(df_i, disp=True, verbose=False)
        summary_i["league"] = league
        summary_i["season"] = season
        summary_i["file"] = os.path.basename(path)
        results_all.append(summary_i)
        tp_i.to_csv(os.path.join(OUT_DIR, f"team_params_{tag}.csv"), index=False)
        gp_i.to_csv(os.path.join(OUT_DIR, f"global_params_{tag}.csv"), index=False)
        st_i.to_csv(os.path.join(OUT_DIR, f"state_table_{tag}.csv"), index=False)
        state_occupancy(prepare_matches(df_i)).to_csv(os.path.join(OUT_DIR, f"state_occupancy_{tag}.csv"), index=False)
    except Exception as e:
        print("失敗:", e)

summary_all_df = pd.DataFrame(results_all)
summary_all_df.to_csv(os.path.join(OUT_DIR, "summary_all.csv"), index=False)
summary_all_df

11 件のcsvが見つかりました
../statsbomb_data/FA_Women's_Super_League/WSL2018-2019_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-99.3314 success=False
[stage2: Powell]   loglik=-1000000105.1849 success=True
[selected: L-BFGS-B] loglik=-99.3314
../statsbomb_data/FA_Women's_Super_League/WSL2019-2020_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-61.9630 success=False
[stage2: Powell]   loglik=-60.4748 success=True
../statsbomb_data/FA_Women's_Super_League/WSL2020-2021_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-90.9814 success=True
[stage2: Powell]   loglik=-90.9608 success=True
../statsbomb_data/FA_Women's_Super_League/WSL2023-2024_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-82.6534 success=True
[stage2: Powell]   loglik=-82.6534 success=True
../statsbomb_data/Frauen_Bundesliga/FB2023-2024_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-96.1287 success=True
[stage2: Powell]   loglik=-4000000121.5219 success=True
[selected: L-BFGS-B] loglik=-96.1287
../statsbomb_data/India

,n_matches,n_teams,n_params,log_likelihood,AIC,BIC,converged,message,league,season,file
0,107,11,33,-99.331436,264.662872,352.866224,True,Optimization terminated successfully.,FA Women's Super League,2018/2019,WSL2018-2019_goals_and_red_cards.csv
1,87,12,35,-60.474794,190.949587,277.256371,True,Optimization terminated successfully.,FA Women's Super League,2019/2020,WSL2019-2020_goals_and_red_cards.csv
2,131,12,35,-90.960772,251.921544,352.553450,True,Optimization terminated successfully.,FA Women's Super League,2020/2021,WSL2020-2021_goals_and_red_cards.csv
3,132,12,35,-82.653359,235.306718,336.204785,True,Optimization terminated successfully.,FA Women's Super League,2023/2024,WSL2023-2024_goals_and_red_cards.csv
4,132,12,35,-96.128681,262.257363,363.155430,True,Optimization terminated successfully.,Frauen Bundesliga,2023/2024,FB2023-2024_goals_and_red_cards.csv
5,115,11,33,-120.677304,307.354608,397.937368,True,Optimization terminated successfully.,Indian Super league,2021/2022,ISL2021-2022_goals_and_red_cards.csv
6,380,20,51,-521.111728,1144.223457,1345.172191,True,Optimization terminated successfully.,La Liga,2015/2016,LL2015-2016_goals_and_red_cards.csv
7,240,16,43,-161.373475,408.746950,558.414423,True,Optimization terminated successfully.,Liga F,2023/2024,LF2023-2024_goals_and_red_cards.csv
8,377,20,51,-581.250889,1264.501778,1465.046282,True,Optimization terminated successfully.,Ligue 1,2015/2016,L12015-2016_goals_and_red_cards.csv
9,380,20,51,-555.579454,1213.158907,1414.107641,True,Optimization terminated successfully.,Premier League,2015/2016,PL2015-2016_goals_and_red_cards.csv
